In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Embedding, Flatten, Concatenate, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

# --- 1. Data Loading & Merging ---
print("Loading data...")
train_df = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/train.csv", parse_dates=['date'])
test_df = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/test.csv", parse_dates=['date'])
stores_df = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/stores.csv")
oil_df = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/oil.csv", parse_dates=['date'])
holidays_df = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv", parse_dates=['date'])

# Merge dataframes
print("Merging data...")
train_df = train_df.merge(stores_df, on='store_nbr', how='left')
test_df = test_df.merge(stores_df, on='store_nbr', how='left')
oil_df = oil_df.rename(columns={'dcoilwtico': 'oil_price'})
train_df = train_df.merge(oil_df, on='date', how='left')
test_df = test_df.merge(oil_df, on='date', how='left')
train_df['oil_price'].bfill(inplace=True)
train_df['oil_price'].ffill(inplace=True)
test_df['oil_price'].bfill(inplace=True)
test_df['oil_price'].ffill(inplace=True)

# For simplicity, we won't merge holidays in this starter NN, but it's a good next step.

# --- 2. Feature Engineering ---
print("Creating features...")
def create_features(df):
    df['year'] = df.date.dt.year
    df['month'] = df.date.dt.month
    df['day'] = df.date.dt.day
    df['dayofweek'] = df.date.dt.dayofweek
    df['dayofyear'] = df.date.dt.dayofyear
    df['weekofyear'] = df.date.dt.isocalendar().week.astype(int)
    return df

train_df = create_features(train_df)
test_df = create_features(test_df)

# Store original IDs for submission
test_ids = test_df['id']
train_df = train_df.drop('id', axis=1)
test_df = test_df.drop('id', axis=1)

# --- 3. Data Preprocessing for Neural Network ---

# Identify categorical and numerical features
categorical_features = ['store_nbr', 'family', 'city', 'state', 'type']
numerical_features = ['onpromotion', 'oil_price', 'year', 'month', 'day', 'dayofweek', 'dayofyear', 'weekofyear']

# Apply Label Encoding to categorical features
for col in categorical_features:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col] = le.transform(test_df[col])

# Scale numerical features
scaler = MinMaxScaler()
train_df[numerical_features] = scaler.fit_transform(train_df[numerical_features])
test_df[numerical_features] = scaler.transform(test_df[numerical_features])

# Prepare data inputs for the model
X_train_cat = [train_df[col].values for col in categorical_features]
X_train_num = train_df[numerical_features].values
y_train_log = np.log1p(train_df['sales'].values)

X_test_cat = [test_df[col].values for col in categorical_features]
X_test_num = test_df[numerical_features].values


# --- 4. Deep Learning Model Building ---
print("Building Deep Learning model...")

inputs = []
embeddings = []

# Create Input and Embedding layers for each categorical feature
for col in categorical_features:
    num_unique_values = train_df[col].nunique()
    embedding_size = min(np.ceil((num_unique_values)/2), 50)
    embedding_size = int(embedding_size)

    cat_input = Input(shape=(1,), name=f'input_{col}')
    embedding_layer = Embedding(num_unique_values, embedding_size, name=f'embedding_{col}')(cat_input)
    flatten_layer = Flatten()(embedding_layer)

    inputs.append(cat_input)
    embeddings.append(flatten_layer)

# Input for numerical features
num_input = Input(shape=(len(numerical_features),), name='input_numerical')
inputs.append(num_input)

# Concatenate all features
concat_layer = Concatenate()(embeddings + [num_input])

# Dense layers
dense1 = Dense(512, activation='relu')(concat_layer)
dropout1 = Dropout(0.5)(dense1)
dense2 = Dense(256, activation='relu')(dropout1)
dropout2 = Dropout(0.5)(dense2)
output = Dense(1, activation='linear')(dropout2) # Linear activation for regression

model = Model(inputs=inputs, outputs=output)
model.compile(optimizer='adam', loss='mean_squared_error')


# --- 5. Model Training ---
print("Training model...")
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.00001)

# We use the last ~10% of data for validation to respect the time series nature
split_index = int(len(X_train_num) * 0.9)

# Split data into train/validation sets
X_train_cat_split = [cat[:split_index] for cat in X_train_cat]
X_val_cat_split = [cat[split_index:] for cat in X_train_cat]
X_train_num_split, X_val_num_split = X_train_num[:split_index], X_train_num[split_index:]
y_train_log_split, y_val_log_split = y_train_log[:split_index], y_train_log[split_index:]

model.fit(X_train_cat_split + [X_train_num_split], y_train_log_split,
          validation_data=(X_val_cat_split + [X_val_num_split], y_val_log_split),
          epochs=5,  # CHANGED TO 5 EPOCHS
          batch_size=1024,
          callbacks=[early_stopping, reduce_lr],
          verbose=1)


# --- 6. Prediction and Submission ---
print("Generating predictions...")
predictions_log = model.predict(X_test_cat + [X_test_num])

# Reverse the log transformation
predictions = np.expm1(predictions_log.flatten())
predictions[predictions < 0] = 0

# Create submission file
submission_df = pd.DataFrame({'id': test_ids, 'sales': predictions})
submission_df.to_csv('submission.csv', index=False)

print("Submission file created successfully!")
print("A preview of the submission file:")
print(submission_df.head())